# Phase 3 — Training and Testing a ML Algorithm
## Dataset: GSE6575 — Gene Expression in Blood of Children with Autism Spectrum Disorder

**Algorithm chosen**: Extra Trees Classifier (Extremely Randomized Trees)
**Task (this version)**: **Binary classification** — `autist` vs `no-autist`.
**Data scarcity strategy**: we enlarge the dataset with **synthetic data augmentation (SMOTE)**, applied **only on the training set** and **inside cross-validation folds**, to avoid data leakage.

**Starting point**: prepared data from Phase 2 (`X_prepared.csv`, `y_labels.csv`).

In [34]:
# -- Library imports ----------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)

# Data augmentation
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # leakage-safe pipeline

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries loaded successfully.')

All libraries loaded successfully.


---
## Step 1 — Setup, Binary Grouping, Split, Reduction & Augmentation

### 1.1 Algorithm: Extra Trees Classifier

**Extra Trees** (Extremely Randomized Trees) builds many unpruned decision trees on the full training set, using **random** split thresholds. Compared to Random Forest it has even lower variance, is faster, is robust to noisy features (useful for microarray data), and provides feature-importance scores.

### 1.2 Binary grouping of the dataset (`autist` vs `no-autist`)

The original `condition` column has 4 classes. For this phase we collapse them into **two distinct groups**:

| Original condition | Binary group |
|---|---|
| Autism no regression | **autist** |
| Autism with regression | **autist** |
| General population | **no-autist** |
| mental retardation or developmental delay | **no-autist** |

In [35]:
# -- Load prepared data from Phase 2 ------------------------------------------
X = pd.read_csv('X_prepared.csv', index_col=0)
y_raw = pd.read_csv('y_labels.csv', index_col=0).squeeze()

# -- 1.2  Binary grouping:  autist  vs  no-autist -----------------------------
AUTIST_CONDITIONS = ['Autism no regression', 'Autism with regression', 'mental retardation or developmental delay']

def to_binary(label):
    return 'autist' if label in AUTIST_CONDITIONS else 'no-autist'

y = y_raw.apply(to_binary)
y.name = 'group'

print('Loaded prepared data:')
print(f'  X shape : {X.shape}  (samples x probes)')
print(f'  y shape : {y.shape}  (binary labels)')
print()
print('Original 4-class distribution:')
print(y_raw.value_counts().to_string())
print()
print('Binary distribution (autist / no-autist):')
print(y.value_counts().to_string())

Loaded prepared data:
  X shape : (56, 54630)  (samples x probes)
  y shape : (56,)  (binary labels)

Original 4-class distribution:
condition
Autism with regression                       18
Autism no regression                         17
General population                           12
mental retardation or developmental delay     9

Binary distribution (autist / no-autist):
group
autist       44
no-autist    12


### 1.3 Train / Test split

The split is **stratified** to keep the autist/no-autist proportions in both sets.
With only 56 samples we use an 80/20 split to maximise training data.

> **Critical rule:** the test set is **never** touched by PCA fitting or by data augmentation. It stays 100% real and unseen, so the final score reflects true generalisation.

In [36]:
# -- 1.3 Train/Test split (stratified, binary) --------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Train/Test split (80/20, stratified):')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print()
print('Class distribution in TRAIN:')
print(y_train.value_counts().to_string())
print()
print('Class distribution in TEST (kept 100% real):')
print(y_test.value_counts().to_string())

Train/Test split (80/20, stratified):
  X_train : (44, 54630)
  X_test  : (12, 54630)

Class distribution in TRAIN:
group
autist       35
no-autist     9

Class distribution in TEST (kept 100% real):
group
autist       9
no-autist    3


### 1.4 Dimensionality reduction with PCA

54,630 features for 56 samples is a severe **p >> n** setting. We reduce dimensionality with **PCA (95% variance)**.

PCA is fitted **on the training set only** and then applied to the test set — fitting it on all the data would leak test information.

We also run augmentation **in PCA space** rather than in the original 54,630-dim space: interpolating between nearest neighbours in a 54,630-dim space with so few points is essentially meaningless (curse of dimensionality), whereas in ~40 PCA dimensions it is much more sensible.

In [37]:
# -- 1.4 PCA: fit on TRAIN only, transform both sets --------------------------
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f'PCA retained {pca.n_components_} components out of {X_train.shape[1]:,} probes')
print(f'Explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}%')
print(f'X_train after PCA: {X_train_pca.shape}')
print(f'X_test  after PCA: {X_test_pca.shape}  (real, untouched)')

PCA retained 38 components out of 54,630 probes
Explained variance: 95.17%
X_train after PCA: (44, 38)
X_test  after PCA: (12, 38)  (real, untouched)


### 1.5 Data augmentation to enlarge the dataset (TRAIN ONLY)

We lack data, so we generate **synthetic samples** with **SMOTE** (Synthetic Minority Over-sampling Technique). SMOTE creates new points by interpolating between a real sample and its nearest neighbours of the same class.

**Two non-negotiable safeguards are applied:**

1. **Augmentation only on the training set**, *after* the split. The test set stays real.
2. PCA + SMOTE are placed **inside a leakage-safe `imblearn` Pipeline** for the cross-validation (section 2.3), so they are re-estimated on each training fold — never on the validation fold.

⚠️ **Honest reading**: SMOTE does **not** create new biological information — it only rebalances and smooths the data you already have. It helps the classifier and balances the classes, but it does **not** replace having more real patients. Trust the scores measured on the **real** test set and on in-fold CV, not on the augmented data itself.

`N_PER_CLASS` controls the target size of each class in the augmented training set (set it `>=` the largest class count in the train set).

In [38]:
# -- 1.5 SMOTE augmentation on the TRAINING set (in PCA space) ----------------
# Target number of samples PER CLASS after augmentation.
# Must be >= the current count of the largest training class.
N_PER_CLASS = 60

train_counts = y_train.value_counts()
print('Train counts before augmentation:')
print(train_counts.to_string())

# k_neighbors must be < number of samples in the smallest class
k = min(5, int(train_counts.min()) - 1)

sampling_strategy = {'autist': N_PER_CLASS, 'no-autist': N_PER_CLASS}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=k)

X_train_aug, y_train_aug = smote.fit_resample(X_train_pca, y_train)

print()
print(f'k_neighbors used : {k}')
print('Train counts AFTER augmentation:')
print(pd.Series(y_train_aug).value_counts().to_string())
print()
n_real = X_train_pca.shape[0]
n_synth = X_train_aug.shape[0] - n_real
print(f'Real training samples     : {n_real}')
print(f'Synthetic samples created : {n_synth}')
print(f'Total training samples    : {X_train_aug.shape[0]}')

Train counts before augmentation:
group
autist       35
no-autist     9

k_neighbors used : 5
Train counts AFTER augmentation:
group
autist       60
no-autist    60

Real training samples     : 44
Synthetic samples created : 76
Total training samples    : 120


#### (Optional) Alternative: Gaussian jitter augmentation

If you prefer simple noise-based augmentation instead of (or in addition to) SMOTE,
the helper below creates noisy copies of real samples. Use it **only on the training set** too.
It is provided as an option and is **not** used in the rest of the notebook.

In [39]:
# -- (Optional) Gaussian jitter -- noisy copies of real training samples ------
def augment_gaussian(X_arr, y_arr, n_copies=2, noise_std=0.05, random_state=42):
    """Return X,y with `n_copies` jittered duplicates of each real sample appended."""
    rng = np.random.default_rng(random_state)
    X_arr = np.asarray(X_arr); y_arr = np.asarray(y_arr)
    feat_std = X_arr.std(axis=0, keepdims=True)
    Xs, ys = [X_arr], [y_arr]
    for _ in range(n_copies):
        noise = rng.normal(0.0, noise_std, X_arr.shape) * feat_std
        Xs.append(X_arr + noise); ys.append(y_arr)
    return np.vstack(Xs), np.concatenate(ys)

# Example (not used downstream):
# X_train_jit, y_train_jit = augment_gaussian(X_train_pca, y_train, n_copies=2, noise_std=0.05)
# print('Jitter-augmented train size:', X_train_jit.shape)

---
## Step 2 — Model Training

### 2.1 Train Extra Trees on the augmented training set

In [40]:
# -- 2.1 Train Extra Trees on the AUGMENTED training set ----------------------
et_model = ExtraTreesClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

et_model.fit(X_train_aug, y_train_aug)

print('Extra Trees model trained on augmented data.')
print(f'Number of trees           : {et_model.n_estimators}')
print(f'PCA features used          : {X_train_aug.shape[1]}')
print(f'Training samples (aug.)    : {X_train_aug.shape[0]}')

Extra Trees model trained on augmented data.
Number of trees           : 300
PCA features used          : 38
Training samples (aug.)    : 120


### 2.2 Training fit (sanity check)

In [41]:
# -- 2.2 Fit on the augmented training set ------------------------------------
y_train_pred = et_model.predict(X_train_aug)
train_accuracy = accuracy_score(y_train_aug, y_train_pred)
train_f1 = f1_score(y_train_aug, y_train_pred, pos_label='autist')

print('=== Training fit (augmented set) ===')
print(f'Training Accuracy        : {train_accuracy:.4f}')
print(f'Training F1 (autist)     : {train_f1:.4f}')
print()
print('Note: near-perfect training fit is expected for unpruned Extra Trees.')
print('Real generalisation is judged by CV (2.3) and the test set (Step 3).')

=== Training fit (augmented set) ===
Training Accuracy        : 1.0000
Training F1 (autist)     : 1.0000

Note: near-perfect training fit is expected for unpruned Extra Trees.
Real generalisation is judged by CV (2.3) and the test set (Step 3).


### 2.3 Leakage-safe Cross-Validation (the trustworthy score)

This is the **most reliable** estimate. PCA **and** SMOTE live inside an `imblearn` Pipeline, so for every fold they are fitted on the training part only and the **validation fold stays real and un-augmented**. This is exactly how augmentation must be validated.

In [42]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import KFold

In [ ]:
# -- 2.3 Stratified CV with PCA + SMOTE INSIDE the pipeline -------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_pipeline = ImbPipeline([
    ('pca',   PCA(n_components=0.95, random_state=42)),
    ('smote', SMOTE(sampling_strategy={'autist': N_PER_CLASS, 'no-autist': N_PER_CLASS},
                    random_state=42, k_neighbors=5)),
    ('clf',   ExtraTreesClassifier(
        n_estimators=300, max_features=None,
        class_weight=None, random_state=42, n_jobs=-1))
])


cv_scores = cross_val_score(cv_pipeline, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
cv_f1     = cross_val_score(cv_pipeline, X, y, cv=skf, scoring='f1_weighted', n_jobs=-1)

print('=== 5-Fold Stratified CV (augmentation done in-fold) ===')
print(f'Accuracy per fold : {[f"{s:.3f}" for s in cv_scores]}')
print(f'Mean Accuracy     : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print()
print(f'F1 per fold       : {[f"{s:.3f}" for s in cv_f1]}')
print(f'Mean F1 (weighted): {cv_f1.mean():.4f} +/- {cv_f1.std():.4f}')

=== 5-Fold Stratified CV (augmentation done in-fold) ===
Accuracy per fold : ['0.750', '0.818', '0.818', '0.818', '0.636']
Mean Accuracy     : 0.7682 +/- 0.0710

F1 per fold       : ['0.643', '0.818', '0.736', '0.736', '0.566']
Mean F1 (weighted): 0.6999 +/- 0.0871


---
## Step 3 — Evaluation on the REAL Test Set

### 3.1 Predictions on the untouched test set

In [ ]:
# -- 3.1 Predict on the REAL, untouched test set ------------------------------
y_pred = et_model.predict(X_test_pca)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1       = f1_score(y_test, y_pred, pos_label='autist')

print('=== Performance on REAL Test Set ===')
print(f'Test Accuracy        : {test_accuracy:.4f}')
print(f'Test F1 (autist)     : {test_f1:.4f}')
print()
print('Comparison:')
print(f'  Training (aug.)    : {train_accuracy:.4f}')
print(f'  CV mean            : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'  Test (real)        : {test_accuracy:.4f}')

### 3.2 Classification report

In [ ]:
# -- 3.2 Detailed classification report ---------------------------------------
print('=== Classification Report (Real Test Set) ===')
print(classification_report(y_test, y_pred, zero_division=0))

### 3.3 Confusion matrix

In [ ]:
# -- 3.3 Confusion matrix -----------------------------------------------------
labels = ['no-autist', 'autist']
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
    ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix -- Extra Trees (Real Test Set)')
plt.tight_layout()
plt.show()

### 3.4 Cross-validation score distribution

In [ ]:
# -- 3.4 CV score distribution ------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([cv_scores, cv_f1], labels=['Accuracy', 'F1 (weighted)'])
ax.scatter(np.ones_like(cv_scores), cv_scores, color='navy', alpha=0.6, zorder=3)
ax.scatter(np.full_like(cv_f1, 2), cv_f1, color='goldenrod', alpha=0.6, zorder=3)
ax.set_title('5-Fold CV score distribution (augmentation in-fold)')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

### 3.5 Analysis of Results

Key points to keep in mind when interpreting these numbers:

- **The CV score (2.3) and the real test score (3.1) are the only trustworthy metrics.** They are measured on real, un-augmented samples.
- The training fit (2.2) is near-perfect by construction (unpruned trees on augmented data) and must **not** be reported as performance.
- SMOTE **balanced and enlarged** the training set but did **not** add new biological information; with n = 56 real patients, modest scores are expected and honest.
- To genuinely improve performance, the highest-leverage levers are: more real samples, supervised feature selection of discriminative probes, or simpler/regularised models — not larger `N_PER_CLASS`.

In [ ]:
# -- 3.6 Final summary --------------------------------------------------------
print('=== Phase 3 Summary (binary: autist vs no-autist) ===')
print()
print(f'Algorithm           : Extra Trees Classifier')
print(f'Task                : Binary (autist vs no-autist)')
print(f'Dimensionality      : {X_train.shape[1]:,} probes -> {pca.n_components_} PCA components (95% var)')
print(f'Train/Test split    : 80/20 stratified')
print(f'Real train samples  : {X_train.shape[0]}  | Synthetic added: {X_train_aug.shape[0]-X_train.shape[0]}')
print(f'Test samples (real) : {X_test.shape[0]}')
print()
print(f'CV Accuracy (mean)  : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}   <- trust this')
print(f'Test Accuracy (real): {test_accuracy:.4f}')
print(f'Test F1 (autist)    : {test_f1:.4f}')
print()
print('Next step -> Phase 4: Ensemble methods / feature selection')